# FinFact-BD XNLI Quality Filter

Uses `joeddav/xlm-roberta-large-xnli` to validate that each perturbed article
is a **meaningful** misinformation sample (contradiction) vs a no-op edit (entailment).

**Setup:**
1. Upload `finfact_bd_perturbed.csv.zst` as a Kaggle Dataset
2. Add that dataset to this notebook's input
3. Run all cells
4. Download `finfact_bd_perturbed_filtered.csv` from output

In [ ]:
!pip install zstandard -q

In [ ]:
import csv, io, json, time, zstandard as zstd
from pathlib import Path
from collections import Counter
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Auto-detect Kaggle paths
INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")

# Find the uploaded dataset
def find_input_file():
    for d in INPUT_DIR.iterdir():
        for f in d.iterdir():
            if "perturbed" in f.name and f.suffix == ".csv":
                return f
            if "perturbed" in f.name and f.suffix == ".zst":
                return f
    raise FileNotFoundError("Upload finfact_bd_perturbed.csv.zst as a Kaggle dataset")

INPUT_FILE = find_input_file()
print(f"Input: {INPUT_FILE}")

In [ ]:
# Load perturbed samples
def load_samples(path):
    samples = []
    if path.suffix == ".zst":
        with open(path, "rb") as f:
            dctx = zstd.ZstdDecompressor()
            with dctx.stream_reader(f) as reader:
                text_stream = io.TextIOWrapper(reader, encoding="utf-8")
                reader = csv.DictReader(text_stream)
                for row in reader:
                    samples.append(row)
    else:
        with open(path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                samples.append(row)
    return samples

perturbed = load_samples(INPUT_FILE)
print(f"Loaded {len(perturbed)} perturbed samples")
print(f"Columns: {list(perturbed[0].keys())}")

In [ ]:
# Load XNLI model
MODEL_NAME = "joeddav/xlm-roberta-large-xnli"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32
MAX_LEN = 512
CONTRADICTION_THRESHOLD = 0.4

print(f"Loading {MODEL_NAME} on {DEVICE}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)
model.eval()
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")
LABEL_MAP = {0: "contradiction", 1: "neutral", 2: "entailment"}

In [ ]:
def score_batch(pairs):
    inputs = tokenizer(
        [p[0] for p in pairs],
        [p[1] for p in pairs],
        padding=True, truncation=True,
        max_length=MAX_LEN, return_tensors="pt",
    ).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=-1)
    results = []
    for i in range(len(pairs)):
        prob_dict = {LABEL_MAP[j]: probs[i][j].item() for j in range(3)}
        results.append({
            "label": LABEL_MAP[probs[i].argmax().item()],
            "probs": prob_dict,
        })
    return results

# Build (original, perturbed) pairs
pairs = [(row["original_text"], row["text"]) for row in perturbed]
print(f"Scoring {len(pairs)} pairs in batches of {BATCH_SIZE}...")

all_results = []
t0 = time.time()
for i in range(0, len(pairs), BATCH_SIZE):
    batch = pairs[i : i + BATCH_SIZE]
    results = score_batch(batch)
    all_results.extend(results)
    if (i // BATCH_SIZE) % 20 == 0:
        pct = min(100, 100 * (i + len(batch)) / len(pairs))
        elapsed = time.time() - t0
        rate = (i + len(batch)) / elapsed
        eta = (len(pairs) - i - len(batch)) / rate if rate > 0 else 0
        print(f"  [{pct:.0f}%] {i+len(batch)}/{len(pairs)} — {elapsed:.0f}s elapsed, ~{eta:.0f}s remaining")

print(f"\nDone in {time.time()-t0:.0f}s")

In [ ]:
# Analysis
label_counts = Counter(r["label"] for r in all_results)
print("Overall distribution:")
for label, count in label_counts.most_common():
    print(f"  {label}: {count} ({100*count/len(all_results):.1f}%)")

# Per-type analysis
type_results = {}
for row, result in zip(perturbed, all_results):
    ptype = row["perturbation_type"]
    type_results.setdefault(ptype, Counter())[result["label"]] += 1

print("\nPer-type distribution:")
for ptype, counts in sorted(type_results.items()):
    total = sum(counts.values())
    contra = 100 * counts.get("contradiction", 0) / total
    entail = 100 * counts.get("entailment", 0) / total
    print(f"  {ptype}: contradiction={contra:.0f}% entailment={entail:.0f}% (n={total})")

In [ ]:
# Filter: keep samples with contradiction probability >= threshold
print(f"Filtering with contradiction threshold >= {CONTRADICTION_THRESHOLD}...")
kept = []
discarded = 0
for row, result in zip(perturbed, all_results):
    contra_prob = result["probs"]["contradiction"]
    if contra_prob >= CONTRADICTION_THRESHOLD:
        row["xnli_contradiction_prob"] = f"{contra_prob:.4f}"
        row["xnli_label"] = result["label"]
        kept.append(row)
    else:
        discarded += 1

print(f"Kept: {len(kept)}, Discarded: {discarded} ({100*discarded/len(perturbed):.1f}%)")

kept_types = Counter(row["perturbation_type"] for row in kept)
print("\nKept per type:")
for ptype, count in kept_types.most_common():
    print(f"  {ptype}: {count}")

In [ ]:
# Save filtered perturbed
out_path = OUTPUT_DIR / "finfact_bd_perturbed_filtered.csv"
fieldnames = list(kept[0].keys())
with open(out_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(kept)
print(f"Saved filtered perturbed to {out_path}")

# Save scores for analysis
scores = []
for row, result in zip(perturbed, all_results):
    scores.append({
        "article_id": row["article_id"],
        "perturbation_type": row["perturbation_type"],
        "xnli_label": result["label"],
        "contradiction_prob": result["probs"]["contradiction"],
        "neutral_prob": result["probs"]["neutral"],
        "entailment_prob": result["probs"]["entailment"],
    })
scores_path = OUTPUT_DIR / "xnli_scores.json"
with open(scores_path, "w", encoding="utf-8") as f:
    json.dump(scores, f, indent=2, ensure_ascii=False)
print(f"Saved XNLI scores to {scores_path}")

# Summary
contra_probs = [r["probs"]["contradiction"] for r in all_results]
print(f"\nContradiction probability stats:")
print(f"  Mean: {sum(contra_probs)/len(contra_probs):.3f}")
print(f"  Median: {sorted(contra_probs)[len(contra_probs)//2]:.3f}")
print(f"  Min: {min(contra_probs):.3f}")
print(f"  Max: {max(contra_probs):.3f}")

In [ ]:
# Download instructions
print("="*60)
print("DONE! Download these files from /kaggle/working/:")
print(f"  1. finfact_bd_perturbed_filtered.csv")
print(f"  2. xnli_scores.json")
print("="*60)